# Attention Layer

In [ ]:
class SimpleAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        
        # Linear layers for Q, K, V
        self.W_q = nn.Linear(d_model, d_model, bias = False)
        self.W_k = nn.Linear(d_model, d_model, bias = False)
        self.W_v = nn.Linear(d_model, d_model, bias = False)
        self.W_o = nn.Linear(d_model, d_model, bias = False)
    
    def forward(self, x, mask=None, cache = None, use_cache =  False):
        # x: (batch, seq, d_model) 
        # In infetence will be (1, T, D_MODEL)
        B, T, C = x.shape
        
        # Project to Q, K, V
        Q = self.W_q(x)  # (batch, seq, d_model)
        K = self.W_k(x)
        V = self.W_v(x)

        if caches is not None:
            assert T == 1, "com cache, entra um token por vez"
            K, V = torch.cat([cache[0], K], dim = 1), torch.cat([cache[1], V], dim = 1)
        else: 
            caches = (K, V) if use_cache else None

            

        # ATTN LOGIC

        return attn_out, (caches if use_cache else None)

# Block Layer

In [ ]:
class Block(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = SimpleAttention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.mlp = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(), nn.Linear(4 * d_model, d_model))

    def forward(self, x, cache=None, use_cache=False):
        attn_out, cache = self.attn(self.ln1(x), cache = cache, use_cache = use_cache)

        x = x + attn_out
        x = x + self.mlp(self.ln2(x))
        return x, cache

# Main Layer

In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab, d_model=1024, n_heads=16, n_layers=24, block_size=1024):
        super().__init__()

        # Embedding
        self.tok = nn.Embedding(vocab, d_model)

        # Blocks
        self.blocks = nn.ModuleList(Block(d_model, n_heads) for _ in range(n_layers))

        # Head
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.head.weight = self.tok.weight

    def forward(self, x, caches = None, use_cache=False):
        x = self.tok(x)
        new_caches = []

        for i, block in enumerate(self.blocks):
            cache = caches[i] if caches is not None else None

            x, new_cache = block(
                x,
                cache=cache,
                use_cache=use_cache
            )

            new_caches.append(new_cache)

        logits = self.head(self.ln_f(x))

        return logits, (new_caches if use_cache else None)

# Decoding PART

In [ ]:
def generate(model, input_ids, max_new_tokens, temperature=1.0, top_k=None, use_cache=True):

    model.eval()
    cache = None

    for _ in range(max_new_tokens):

        # If we already have an cache use this
        if use_cache and cache is not None: # If implement MAX TOKENS GENERATING USE THIS (and input_ids.size(1) <= block:)
            # DECODE: só o token novo, na posição certa
            logits, caches = model(input_ids[:, -1:], cache = cache, use_cache=True) # Will pass only the last token of INPUT_IDS
        else:
            # PREFILL (ou recálculo, se passou do contexto)
            logits, caches = model(input_ids, cache = cache, use_cache = use_cache)
